In [1]:
import requests
import numpy as np
import pandas as pd
import time


# --------------------------------------------------
# 1. NASA POWER Funktion
# --------------------------------------------------

def get_nasa_irradiance(lat, lon):
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"

    params = {
        "parameters": "ALLSKY_SFC_SW_DWN",
        "community": "RE",
        "longitude": lon,
        "latitude": lat,
        "start": "20230101",
        "end": "20231231",
        "format": "JSON"
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()

    irradiance = data["properties"]["parameter"]["ALLSKY_SFC_SW_DWN"]

    annual_sum = sum(irradiance.values())

    return annual_sum


# --------------------------------------------------
# 2. Vier umliegende Rasterpunkte bestimmen
# --------------------------------------------------

def get_surrounding_grid_points(lat, lon, step=1.0):
    lat_lower = np.floor(lat / step) * step
    lat_upper = lat_lower + step

    lon_lower = np.floor(lon / step) * step
    lon_upper = lon_lower + step

    return {
        "bottom_left": (lat_lower, lon_lower),
        "bottom_right": (lat_lower, lon_upper),
        "top_left": (lat_upper, lon_lower),
        "top_right": (lat_upper, lon_upper),
}


# --------------------------------------------------
# 3. Bilineare Interpolation
# --------------------------------------------------

def interpolate_solar_value(target_lat, target_lon, step=1.0):
    points = get_surrounding_grid_points(
        target_lat,
        target_lon,
        step
    )

    values = {}

    for name, (lat, lon) in points.items():
        print(f"Lade NASA-Daten für {name}: {lat}, {lon}")

        values[name] = get_nasa_irradiance(lat, lon)

        time.sleep(0.3)

    lat_lower, lon_lower = points["bottom_left"]
    lat_upper, lon_upper = points["top_right"]

    q11 = values["bottom_left"]
    q21 = values["bottom_right"]
    q12 = values["top_left"]
    q22 = values["top_right"]

    interpolated_value = (
        q11 * (lat_upper - target_lat) * (lon_upper - target_lon) +
        q21 * (lat_upper - target_lat) * (target_lon - lon_lower) +
        q12 * (target_lat - lat_lower) * (lon_upper - target_lon) +
        q22 * (target_lat - lat_lower) * (target_lon - lon_lower)
    ) / (
        (lat_upper - lat_lower) * (lon_upper - lon_lower)
    )

    return interpolated_value, points, values


# --------------------------------------------------
# 4. Eingabe
# --------------------------------------------------

target_lat = float(input("Latitude eingeben: "))
target_lon = float(input("Longitude eingeben: "))

solar_value, grid_points, grid_values = interpolate_solar_value(
    target_lat,
    target_lon,
    step=1.0
)

print("\n--------------------------------")
print("Four used NASA-Rasterpoints")
print("--------------------------------")

for name, point in grid_points.items():
    print(f"{name}: {point} -> {grid_values[name]:.2f} kWh/m²/year")

print("\n--------------------------------")
print("Interpolated sun exposure value")
print("--------------------------------")
print(f"Latitude: {target_lat}")
print(f"Longitude: {target_lon}")
print(f"Sun exposure: {solar_value:.2f} kWh/m²/year")

Lade NASA-Daten für bottom_left: 48.0, 11.0
Lade NASA-Daten für bottom_right: 48.0, 12.0
Lade NASA-Daten für top_left: 49.0, 11.0
Lade NASA-Daten für top_right: 49.0, 12.0

--------------------------------
Four used NASA-Rasterpoints
--------------------------------
bottom_left: (np.float64(48.0), np.float64(11.0)) -> 1208.38 kWh/m²/year
bottom_right: (np.float64(48.0), np.float64(12.0)) -> 1197.78 kWh/m²/year
top_left: (np.float64(49.0), np.float64(11.0)) -> 1159.85 kWh/m²/year
top_right: (np.float64(49.0), np.float64(12.0)) -> 1146.90 kWh/m²/year

--------------------------------
Interpolated sun exposure value
--------------------------------
Latitude: 48.1351
Longitude: 11.582
Sun exposure: 1195.47 kWh/m²/year
